In [1]:
import os
from sdv.single_table import CTGANSynthesizer, GaussianCopulaSynthesizer, TVAESynthesizer
from sdv.metadata import Metadata

In [ ]:
SAMPLE_SIZES = (10_000, 100_000, 1_000_000)
METADATA_FILE = "spoofing_metadata.json"


In [ ]:
from gps_ids import protocol, schema, synthetic

_schema = schema.get('2d')
X_train_raw, y_train = protocol.real_train_rows(_schema.key)
df = X_train_raw.copy()
df[_schema.label_column] = y_train
CLASSES = sorted(set(y_train.tolist()))


In [4]:
# Load or create metadata
def _columns_of(meta):
    d = meta.to_dict()
    table = list(d['tables'].values())[0] if 'tables' in d else d
    return set(table['columns'])


def _detect(frame):
    meta = Metadata()
    meta.detect_table_from_dataframe(data=frame, table_name='spoofing_data')
    return meta


metadata = None
if os.path.exists(METADATA_FILE):
    cached = Metadata.load_from_json(METADATA_FILE)
    if _columns_of(cached) == set(df.columns):
        metadata = cached
        print(f"Loaded metadata from {METADATA_FILE}")
    else:
        print(f"{METADATA_FILE} does not describe this dataframe; re-detecting.")

if metadata is None:
    metadata = _detect(df)
    metadata.save_to_json(METADATA_FILE, mode='overwrite')
    print(f"Created and saved new metadata to {METADATA_FILE}")

Loaded metadata from spoofing_metadata.json


In [5]:
# Synthesizer configurations
# If you don't have a GPU available, it will automatically use CPU.
synthesizers = [
    {
        'name': 'CTGAN',
        'generator': 'ctgan',
        'class': CTGANSynthesizer,
        'params': {
            'enforce_min_max_values': True,
            'enforce_rounding': False,
            'epochs': 4000,
            'verbose': True,
            'cuda': True
        },
        'pickle': "spoofing-CTGAN-synthesizer.pkl"
    },
    {
        'name': 'GaussianCopula',
        'generator': 'gc',
        'class': GaussianCopulaSynthesizer,
        'params': {
            'enforce_min_max_values': True,
            'enforce_rounding': False,
        },
        'pickle': "spoofing-GaussianCopula-synthesizer.pkl"
    },
    {
        'name': 'TVAE',
        'generator': 'tvae',
        'class': TVAESynthesizer,
        'params': {
            'enforce_min_max_values': True,
            'enforce_rounding': False,
            'epochs': 2000,
            'verbose': True,
            'cuda': True
        },
        'pickle': "spoofing-TVAE-synthesizer.pkl"
    }
]

In [ ]:
for synth in synthesizers:
    print(f"\nProcessing {synth['name']}...")
    SynthClass = synth['class']
    synthesizer = SynthClass(metadata, **synth['params'])
    synthesizer.fit(df)
    synthesizer.save(filepath=synth['pickle'])
    print(f"Trained synthesizer saved to {synth['pickle']}")
    for size in SAMPLE_SIZES:
        synthetic_data = synthetic.sample_balanced(
            synthesizer, _schema.label_column, CLASSES, size, seed=protocol.SEED
        )
        out_path = schema.synthetic_path(
            _schema, f"spoofing_{synth['generator']}_{size}.csv"
        )
        synthetic_data.to_csv(out_path, index=False)
        balance = synthetic.class_balance(synthetic_data, _schema.label_column)
        print(f"{synth['name']} x{size}: {synthetic_data.shape} {balance}")
        print(f"Synthetic dataset saved to {out_path}")
